In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install optuna
!pip install kagglehub
!pip install cuml-cu12
!pip install optuna-integration

Tuning and Training Ridge Regression algorithm

In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import time
import joblib
import kagglehub

# Download the normalized taxi data and map the file paths to their corresponding dataset names
path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {
    f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
    f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
    f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
    f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
    f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
    f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
    f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
    f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"
}

# Define boundaries for the cross-validation folds and the final holdout test set
first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

# Set up SQLite database to persist Optuna hyperparameter tuning history
DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Ridge Regression/Ridge_Regression_tuning.db"

all_datasets_results = []

for file in datasets:
    dataset_name = datasets[file]

    # Load and clean the data, retaining only 2023-2024 records and dropping irrelevant features
    df = pd.read_parquet(file)
    df = df.rename(columns={df.columns[1]:"date"})
    df = df[df["year"].isin([2023, 2024])]
    df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
    df = df.drop(columns=["precipitation"], errors='ignore')
    df = df.drop(columns=["total_amount"], errors="ignore")
    df = df.dropna()

    # Split data into development (tuning/training) and holdout test sets based on the 2024 threshold
    df_test = df[df["date"]>=test_set_start].copy()
    df_dev = df[df["date"]<test_set_start].copy()

    X_test = df_test.drop(columns=["trip_count","date","year"])
    y_test = df_test["trip_count"]

    X_dev = df_dev.drop(columns=["trip_count","date","year"])
    y_dev = df_dev["trip_count"]

    # Construct a custom rolling-window cross-validation strategy (AI generated):
    # Trains on 6 months of data and validates on the subsequent 1 month, sliding forward 6 times.
    custom_folds = []
    for month in range(6):
        val_month_start = first_block_end + pd.DateOffset(months=month)
        val_month_end = val_month_start + pd.DateOffset(months=1)

        train_month_start = val_month_start - pd.DateOffset(months=6)

        # Use searchsorted to efficiently find integer index positions for the date boundaries
        train_start_i = df_dev["date"].searchsorted(train_month_start)
        train_end_i = df_dev["date"].searchsorted(val_month_start)
        val_end_i = df_dev["date"].searchsorted(val_month_end)

        train_indices = np.arange(train_start_i,train_end_i)
        val_indices = np.arange(train_end_i,val_end_i)

        custom_folds.append((train_indices,val_indices))

    def objective(trial):
        # Define the search space for the regularization strength (alpha) on a logarithmic scale
        alpha = trial.suggest_float("alpha", 1e-7, 1e3, log=True)

        model = Ridge(alpha=alpha)
        fold_errors = []

        # Evaluate the current alpha across all custom time-based folds (AI generated)
        for train_i, val_i in custom_folds:
            X_train, X_val = X_dev.iloc[train_i], X_dev.iloc[val_i]
            y_train, y_val = y_dev.iloc[train_i], y_dev.iloc[val_i]

            model.fit(X_train, y_train)
            pred = model.predict(X_val)

            mae = mean_absolute_error(y_val,pred)
            fold_errors.append(mae)

        # Return the average Mean Absolute Error to guide the Optuna optimizer
        return np.mean(fold_errors)

    # Initialize the Optuna study, pointing it to the SQLite DB to allow resuming interrupted studies
    study = optuna.create_study(
        study_name=f"ridge_{dataset_name}",
        storage=DataBase_URL,
        load_if_exists=True,
        direction="minimize"
    )

    # Execute 80 hyperparameter tuning trials and track execution time
    tuning_start = time.perf_counter()
    study.optimize(objective, n_trials=80)
    tuning_time = time.perf_counter() - tuning_start

    best_alpha = study.best_params["alpha"]
    print(f"Tuning hyperparameters for {dataset_name}:{best_alpha}")

    # Train the final model on the entire development dataset using the optimized alpha
    final_model = Ridge(alpha=best_alpha)
    training_start = time.perf_counter()
    final_model.fit(X_dev, y_dev)
    training_time = time.perf_counter() - training_start

    # Serialize and save the fully trained model for later inference
    joblib.dump(final_model, f"drive/MyDrive/BT_Florian_2026/Trained Models/Ridge Regression/{dataset_name}_model.pkl")

    # Evaluate the final model on the 2024 holdout test set and record performance metrics
    testing_start = time.perf_counter()
    test_preds = final_model.predict(X_test)
    final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    final_test_mae = mean_absolute_error(y_test,test_preds)
    testing_time = time.perf_counter() - testing_start

    # Store all execution metrics and performance results for this dataset loop
    result_dict = {
        "Dataset": [dataset_name],
        "RMSE": [final_test_rmse],
        "MAE": [final_test_mae],
        "Tuning_Time_sec": [tuning_time],
        "Training_Time_sec": [training_time],
        "Testing_Time_sec": [testing_time]
    }

    all_datasets_results.append(result_dict)

# Consolidate all metrics into a single DataFrame and export to CSV
df_final_results = pd.DataFrame(all_datasets_results)
df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Ridge Regression/RidgeRegressionResults.csv", index=False)

Tuning and Training Random Forrest Regression algorithm

In [ ]:
import pandas as pd
import numpy as np
import optuna
from cuml.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import time
import joblib
import kagglehub

# Download the normalized dataset and map parquet file paths to dataset names
path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {
    f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
    f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
    f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
    f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
    f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
    f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
    f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
    f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"
}

# Define cross-validation and holdout test set temporal boundaries
first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

# SQLite database for tracking and resuming Optuna hyperparameter tuning
DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Random Forrest/Random_Forrest_tuning.db"

all_datasets_results = []

for file in datasets:
    dataset_name = datasets[file]

    # Load and preprocess data: rename date, map years to binary, and drop unneeded features
    df = pd.read_parquet(file)
    df = df.rename(columns={df.columns[1]:"date"})
    df = df[df["year"].isin([2023, 2024])]
    df["year"] = df["year"].map({2023: 0, 2024: 1})
    df = df.drop(columns=["DOLocationID", "PULocationID", "precipitation", "total_amount"], errors="ignore")
    df = df.dropna()

    # Split into development (2023) and holdout test (2024) sets
    df_test = df[df["date"]>=test_set_start].copy()
    df_dev = df[df["date"]<test_set_start].copy()

    X_test = df_test.drop(columns=["trip_count","date"])
    y_test = df_test["trip_count"]

    X_dev = df_dev.drop(columns=["trip_count","date"])
    y_dev = df_dev["trip_count"]

    # Build custom rolling-window cross-validation folds for time-series evaluation (AI generated)
    # Trains on a 6-month window, validates on the 1 subsequent month, repeating 6 times
    custom_folds = []
    for month in range(6):
        val_month_start = first_block_end + pd.DateOffset(months=month)
        val_month_end = val_month_start + pd.DateOffset(months=1)
        train_month_start = val_month_start - pd.DateOffset(months=6)

        train_start_i = df_dev["date"].searchsorted(train_month_start)
        train_end_i = df_dev["date"].searchsorted(val_month_start)
        val_end_i = df_dev["date"].searchsorted(val_month_end)

        train_indices = np.arange(train_start_i, train_end_i)
        val_indices = np.arange(train_end_i, val_end_i)

        custom_folds.append((train_indices, val_indices))

    def objective(trial):
        # Define the hyperparameter search space for the Random Forest
        n_estimators = trial.suggest_int("n_estimators", 25, 100)
        max_depth = trial.suggest_int("max_depth", 10, 25)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 100, 1000, log=True)
        max_features = trial.suggest_float("max_features", 0.25, 0.75)
        max_samples = trial.suggest_float("max_samples", 0.1, 0.5)

        # Initialize the RAPIDS cuML GPU-accelerated Random Forest
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            max_samples=max_samples,
            random_state=100
        )

        fold_errors = []

        # Evaluate model performance across all time-series folds (AI generated)
        for train_i, val_i in custom_folds:
            X_train, X_val = X_dev.iloc[train_i], X_dev.iloc[val_i]
            y_train, y_val = y_dev.iloc[train_i], y_dev.iloc[val_i]

            model.fit(X_train, y_train)
            pred = model.predict(X_val)

            mae = mean_absolute_error(y_val, pred)
            fold_errors.append(mae)

        # Return the cross-validated Mean Absolute Error to guide Optuna
        return np.mean(fold_errors)

    # Initialize and run the Optuna optimization study (50 trials)
    study = optuna.create_study(
        study_name=f"RF_{dataset_name}",
        storage=DataBase_URL,
        load_if_exists=True,
        direction="minimize"
    )

    tuning_start = time.perf_counter()
    study.optimize(objective, n_trials=50)
    tuning_time = time.perf_counter() - tuning_start

    best_params = study.best_params
    print(f"Best hyperparameters for {dataset_name}: {best_params}")

    # Retrain the final model on the full development set using the optimal hyperparameters
    final_model = RandomForestRegressor(**best_params, random_state=100)

    training_start = time.perf_counter()
    final_model.fit(X_dev, y_dev)
    training_time = time.perf_counter() - training_start

    # Save the trained Random Forest model to disk
    joblib.dump(final_model, f"drive/MyDrive/BT_Florian_2026/Trained Models/Random Forrest/{dataset_name}_model.pkl")

    # Generate test predictions and calculate final evaluation metrics
    testing_start = time.perf_counter()
    test_preds = final_model.predict(X_test)
    final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    final_test_mae = mean_absolute_error(y_test, test_preds)
    testing_time = time.perf_counter() - testing_start

    # Record the performance and execution time for the current dataset
    result_dict = {
        "Dataset": [dataset_name],
        "RMSE": [final_test_rmse],
        "MAE": [final_test_mae],
        "Tuning_Time_sec": [tuning_time],
        "Training_Time_sec": [training_time],
        "Testing_Time_sec": [testing_time]
    }

    all_datasets_results.append(result_dict)

# Export all aggregated dataset metrics to a CSV file
df_final_results = pd.DataFrame(all_datasets_results)
df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Random Forrest/RandomForrestResults.csv", index=False)

Tuning and Training Feed-Forward Neural Network models

In [ ]:
import os
import pandas as pd
import numpy as np
import time
import optuna
from optuna_integration import TFKerasPruningCallback
import kagglehub
import tensorflow as tf
from tensorflow.keras import mixed_precision
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Enable mixed precision (float16) to accelerate training on compatible GPUs and reduce memory usage
mixed_precision.set_global_policy("mixed_float16")

# Download the normalized dataset and map parquet file paths to dataset names
path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {
    f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
    f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
    f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt",
    f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
    f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
    f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
    f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt",
    f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"
}

# Define cross-validation and holdout test set temporal boundaries
first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

# SQLite database for persistent tracking of Optuna hyperparameter tuning
DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/FFNN_tuning.db"

all_datasets_results = []

for file in datasets:
    dataset_name = datasets[file]

    # Load and preprocess data: rename date, map years to binary, and drop unneeded features
    df = pd.read_parquet(file)
    df = df.rename(columns={df.columns[1]:"date"})
    df = df[df["year"].isin([2023, 2024])]
    df["year"] = df["year"].map({2023: 0, 2024: 1})
    df = df.drop(columns=["DOLocationID", "PULocationID", "precipitation", "total_amount"], errors="ignore")
    df = df.dropna()

    # Split into development (2023) and holdout test (2024) sets
    df_test = df[df["date"]>=test_set_start].copy()
    df_dev = df[df["date"]<test_set_start].copy()

    X_test = df_test.drop(columns=["trip_count","date"])
    y_test = df_test["trip_count"]

    X_dev = df_dev.drop(columns=["trip_count","date"])
    y_dev = df_dev["trip_count"]

    # Convert the development DataFrames into TensorFlow tensors for faster Keras processing
    X_dev = tf.constant(X_dev.values, dtype=tf.float32)
    y_dev = tf.constant(y_dev.values, dtype=tf.float32)

    # Build custom rolling-window cross-validation folds for time-series evaluation (AI generated)
    # Trains on a 6-month window, validates on the 1 subsequent month, repeating 6 times
    custom_folds = []
    for month in range(6):
        val_month_start = first_block_end + pd.DateOffset(months=month)
        val_month_end = val_month_start + pd.DateOffset(months=1)
        train_month_start = val_month_start - pd.DateOffset(months=6)

        train_start_i = df_dev["date"].searchsorted(train_month_start)
        train_end_i = df_dev["date"].searchsorted(val_month_start)
        val_end_i = df_dev["date"].searchsorted(val_month_end)

        train_indices = np.arange(train_start_i, train_end_i)
        val_indices = np.arange(train_end_i, val_end_i)

        custom_folds.append((train_indices, val_indices))

    def objective(trial):
        # Define the hyperparameter search space for the Neural Network topology and learning dynamics
        hidden_layers = trial.suggest_int("hidden_layers", 1, 5)
        neurons_per_layer = trial.suggest_categorical("neurons_per_layer", [16, 32, 64])
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [1024, 2048, 4096])
        dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)

        fold_epochs = []
        fold_errors = []

        # Evaluate the current architecture across all time-series folds (AI generated for loop)
        for fold_id, (train_i, val_i) in enumerate(custom_folds):
            # Clear backend to free GPU memory between folds
            tf.keras.backend.clear_session()

            # Slice the tensors according to the fold indices (AI generated)
            X_train, X_val = tf.gather(X_dev, train_i), tf.gather(X_dev, val_i)
            y_train, y_val = tf.gather(y_dev, train_i), tf.gather(y_dev, val_i)

            # Dynamically construct the Feed-Forward Neural Network (AI generated)
            model = tf.keras.Sequential()
            model.add(tf.keras.Input(shape=(X_train.shape[1],)))

            for i in range(hidden_layers):
                model.add(tf.keras.layers.Dense(neurons_per_layer))
                model.add(tf.keras.layers.BatchNormalization())
                model.add(tf.keras.layers.Activation("relu"))
                model.add(tf.keras.layers.Dropout(dropout_rate))

            # Linear output layer for regression
            model.add(tf.keras.layers.Dense(1, activation="linear"))

            model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="mae")

            # Setup EarlyStopping, and add Optuna's Pruning Callback only on the first fold to halt bad trials early (AI generated)
            early_stopper = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
            callbacks = [early_stopper]
            if fold_id == 0:
                callbacks.append(TFKerasPruningCallback(trial, "val_loss"))

            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                batch_size=batch_size,
                epochs=100,
                callbacks=callbacks,
                verbose=0
            )

            # Track the epoch where the model performed best to inform final training
            best_epoch = np.argmin(history.history["val_loss"]) + 1
            fold_epochs.append(best_epoch)

            mae = model.evaluate(X_val, y_val, verbose=0)
            fold_errors.append(mae)

        # Log the average optimal epochs across all folds as an attribute of the trial (AI generated)
        trial.set_user_attr("optimal_epochs", round(np.mean(fold_epochs)))

        # Return the cross-validated Mean Absolute Error to guide Optuna
        return np.mean(fold_errors)

    # Use RAM-based storage for the active tuning process to maximize speed
    ram_storage = optuna.storages.InMemoryStorage()

    study = optuna.create_study(
        study_name=f"NN_{dataset_name}",
        storage=ram_storage,
        load_if_exists=True,
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=10)
    )

    tuning_start = time.perf_counter()
    study.optimize(objective, n_trials=50)
    tuning_time = time.perf_counter() - tuning_start

    # Copy the completed study from RAM into the persistent SQLite database
    optuna.copy_study(
        from_study_name=f"NN_{dataset_name}",
        from_storage=ram_storage,
        to_storage=DataBase_URL,
        to_study_name=f"NN_{dataset_name}"
    )

    best_params = study.best_params

    # Retrieve the average optimal epochs tracked during CV to use for the final model training (AI generated)
    avg_best_epochs = study.best_trial.user_attrs["optimal_epochs"]
    best_params["epochs"] = int(np.round(avg_best_epochs))

    tf.keras.backend.clear_session()

    # Reconstruct the optimal architecture to train on the entire development dataset (AI generated)
    final_model = tf.keras.Sequential()
    final_model.add(tf.keras.Input(shape=(X_dev.shape[1],)))

    for i in range(best_params["hidden_layers"]):
        final_model.add(tf.keras.layers.Dense(best_params["neurons_per_layer"]))
        final_model.add(tf.keras.layers.BatchNormalization())
        final_model.add(tf.keras.layers.Activation("relu"))
        final_model.add(tf.keras.layers.Dropout(best_params["dropout_rate"]))

    final_model.add(tf.keras.layers.Dense(1, activation="linear"))

    final_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["learning_rate"]),
        loss="mae"
    )

    training_start = time.perf_counter()

    # Train the final model using the exact number of optimal epochs determined by CV
    history = final_model.fit(
        X_dev, y_dev,
        batch_size=best_params["batch_size"],
        epochs=best_params["epochs"],
        verbose=0
    )

    training_time = time.perf_counter() - training_start

    # Save the finalized model and its training history to disk
    final_model.save(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/{dataset_name}_model.keras")

    testing_start = time.perf_counter()

    # Generate test predictions and calculate final evaluation metrics
    test_preds = final_model.predict(X_test)
    test_preds = test_preds.flatten()

    final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    final_test_mae = mean_absolute_error(y_test, test_preds)

    testing_time = time.perf_counter() - testing_start

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/{dataset_name}_history.csv", index=False)

    # Record the performance and execution time for the current dataset
    result_dict = {
        "Dataset": [dataset_name],
        "RMSE": [final_test_rmse],
        "MAE": [final_test_mae],
        "Tuning_Time_sec": [tuning_time],
        "Training_Time_sec": [training_time],
        "Testing_Time_sec": [testing_time]
    }

    # Incrementally save the results for the current iteration to prevent data loss in case of a crash
    df_iteration = pd.DataFrame(result_dict)
    csv_path = "drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/NeuralNetworkResults.csv"

    if not os.path.isfile(csv_path):
        df_iteration.to_csv(csv_path, index=False)
    else:
        df_iteration.to_csv(csv_path, mode="a", header=False, index=False)

    all_datasets_results.append(result_dict)

# Export all aggregated dataset metrics globally
df_final_results = pd.DataFrame(all_datasets_results)
df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/NeuralNetworkResults.csv", index=False)

DOs

In [ ]:
import pandas as pd
import numpy as np
import time
import optuna
import os
from optuna_integration import TFKerasPruningCallback
import kagglehub
import tensorflow as tf
from tensorflow.keras import mixed_precision
from sklearn.metrics import mean_absolute_error, mean_squared_error

mixed_precision.set_global_policy("mixed_float16")

path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyDOs202324_Normalized.parquet": "df_DOs",
            f"{path}/hourlyDOs202324WEATHER_Normalized.parquet":"df_DOsWEATHER",
            f"{path}/hourlyDOs202324CYCLt_Normalized.parquet": "df_DOsCYCLt"}


first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/FFNN_tuning.db"

all_datasets_results = []

for file in datasets:
  dataset_name = datasets[file]
  df = pd.read_parquet(file)
  df = df.rename(columns={df.columns[1]:"date"})
  df = df[df["year"].isin([2023, 2024])]
  df["year"] = df["year"].map({2023: 0, 2024: 1})
  df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
  df = df.drop(columns=["precipitation"], errors='ignore')
  df = df.drop(columns=["total_amount"], errors="ignore")
  df = df.dropna()

  df_test = df[df["date"]>=test_set_start].copy()
  df_dev = df[df["date"]<test_set_start].copy()

  X_test = df_test.drop(columns=["trip_count","date"])
  y_test = df_test["trip_count"]

  X_dev = df_dev.drop(columns=["trip_count","date"])
  y_dev = df_dev["trip_count"]

  X_dev = tf.constant(X_dev.values, dtype=tf.float32)
  y_dev = tf.constant(y_dev.values, dtype=tf.float32)

  custom_folds = []
  for month in range(6):
    val_month_start = first_block_end + pd.DateOffset(months=month)
    val_month_end = val_month_start + pd.DateOffset(months=1)

    train_month_start = val_month_start - pd.DateOffset(months=6)

    train_start_i = df_dev["date"].searchsorted(train_month_start)
    train_end_i = df_dev["date"].searchsorted(val_month_start)
    val_end_i = df_dev["date"].searchsorted(val_month_end)

    train_indices = np.arange(train_start_i,train_end_i)
    val_indices = np.arange(train_end_i,val_end_i)

    custom_folds.append((train_indices,val_indices))

  def objective(trial):
    hidden_layers = trial.suggest_int("hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_categorical("neurons_per_layer", [16, 32, 64])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [1024, 2048, 4096])
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)

    fold_epochs = []
    fold_errors = []

    for fold_id, (train_i, val_i) in enumerate(custom_folds):
      tf.keras.backend.clear_session()

      X_train, X_val = tf.gather(X_dev, train_i), tf.gather(X_dev, val_i)
      y_train, y_val = tf.gather(y_dev, train_i), tf.gather(y_dev, val_i)

      model = tf.keras.Sequential()
      model.add(tf.keras.Input(shape=(X_train.shape[1],)))

      for i in range(hidden_layers):
        model.add(tf.keras.layers.Dense(neurons_per_layer))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation("relu"))
        model.add(tf.keras.layers.Dropout(dropout_rate))

      model.add(tf.keras.layers.Dense(1, activation="linear"))

      model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="mae")

      early_stopper = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
      callbacks = [early_stopper]
      if fold_id == 0:
        callbacks.append(TFKerasPruningCallback(trial, "val_loss"))

      history = model.fit(
          X_train, y_train,
          validation_data=(X_val, y_val),
          batch_size=batch_size,
          epochs=100,
          callbacks=callbacks,
          verbose=0
      )

      best_epoch = np.argmin(history.history["val_loss"]) + 1
      fold_epochs.append(best_epoch)

      mae = model.evaluate(X_val, y_val, verbose=0)
      fold_errors.append(mae)

    trial.set_user_attr("optimal_epochs", round(np.mean(fold_epochs)))

    return np.mean(fold_errors)

  ram_storage = optuna.storages.InMemoryStorage()

  study = optuna.create_study(
      study_name=f"NN_{dataset_name}",
      storage=ram_storage,
      load_if_exists=True,
      direction="minimize",
      pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=10)
  )

  tuning_start = time.perf_counter()
  study.optimize(objective, n_trials=50)
  tuning_time = time.perf_counter() - tuning_start

  optuna.copy_study(
      from_study_name=f"NN_{dataset_name}",
      from_storage=ram_storage,
      to_storage=DataBase_URL,
      to_study_name=f"NN_{dataset_name}"
  )

  best_params = study.best_params

  avg_best_epochs = study.best_trial.user_attrs["optimal_epochs"]
  best_params["epochs"] = int(np.round(avg_best_epochs))

  tf.keras.backend.clear_session()

  final_model = tf.keras.Sequential()
  final_model.add(tf.keras.Input(shape=(X_dev.shape[1],)))

  for i in range(best_params["hidden_layers"]):
    final_model.add(tf.keras.layers.Dense(best_params["neurons_per_layer"]))
    final_model.add(tf.keras.layers.BatchNormalization())
    final_model.add(tf.keras.layers.Activation("relu"))
    final_model.add(tf.keras.layers.Dropout(best_params["dropout_rate"]))

  final_model.add(tf.keras.layers.Dense(1, activation="linear"))

  final_model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["learning_rate"]),
      loss="mae"
  )

  training_start = time.perf_counter()

  history = final_model.fit(
      X_dev, y_dev,
      batch_size=best_params["batch_size"],
      epochs=best_params["epochs"],
      verbose=0
  )

  training_time = time.perf_counter() - training_start

  final_model.save(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/{dataset_name}_model.keras")

  testing_start = time.perf_counter()

  test_preds = final_model.predict(X_test)

  test_preds = test_preds.flatten()

  final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
  final_test_mae = mean_absolute_error(y_test, test_preds)

  testing_time = time.perf_counter() - testing_start

  history_df = pd.DataFrame(history.history)
  history_df.to_csv(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/{dataset_name}_history.csv", index=False)

  result_dict = {
      "Dataset": [dataset_name],
      "RMSE": [final_test_rmse],
      "MAE": [final_test_mae],
      "Tuning_Time_sec": [tuning_time],
      "Training_Time_sec": [training_time],
      "Testing_Time_sec": [testing_time]
  }

  df_iteration = pd.DataFrame(result_dict)
  csv_path = "drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/NeuralNetworkResults.csv"

  if not os.path.isfile(csv_path):
    df_iteration.to_csv(csv_path, index=False)
  else:
    df_iteration.to_csv(csv_path, mode="a", header=False, index=False)

  #all_datasets_results.append(result_dict)

#df_final_results = pd.DataFrame(all_datasets_results)
#df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/NeuralNetworkResults.csv", index=False)

PUs

In [ ]:
import pandas as pd
import numpy as np
import time
import optuna
import os
from optuna_integration import TFKerasPruningCallback
import kagglehub
import tensorflow as tf
from tensorflow.keras import mixed_precision
from sklearn.metrics import mean_absolute_error, mean_squared_error

mixed_precision.set_global_policy("mixed_float16")

path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyPUs202324_Normalized.parquet":"df_PUs",
            f"{path}/hourlyPUs202324WEATHER_Normalized.parquet":"df_PUsWEATHER",
            f"{path}/hourlyPUs202324CYCLt_Normalized.parquet":"df_PUsCYCLt"}


first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/FFNN_tuning.db"

all_datasets_results = []

for file in datasets:
  dataset_name = datasets[file]
  df = pd.read_parquet(file)
  df = df.rename(columns={df.columns[1]:"date"})
  df = df[df["year"].isin([2023, 2024])]
  df["year"] = df["year"].map({2023: 0, 2024: 1})
  df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
  df = df.drop(columns=["precipitation"], errors='ignore')
  df = df.drop(columns=["total_amount"], errors="ignore")
  df = df.dropna()

  df_test = df[df["date"]>=test_set_start].copy()
  df_dev = df[df["date"]<test_set_start].copy()

  X_test = df_test.drop(columns=["trip_count","date"])
  y_test = df_test["trip_count"]

  X_dev = df_dev.drop(columns=["trip_count","date"])
  y_dev = df_dev["trip_count"]

  X_dev = tf.constant(X_dev.values, dtype=tf.float32)
  y_dev = tf.constant(y_dev.values, dtype=tf.float32)

  custom_folds = []
  for month in range(6):
    val_month_start = first_block_end + pd.DateOffset(months=month)
    val_month_end = val_month_start + pd.DateOffset(months=1)

    train_month_start = val_month_start - pd.DateOffset(months=6)

    train_start_i = df_dev["date"].searchsorted(train_month_start)
    train_end_i = df_dev["date"].searchsorted(val_month_start)
    val_end_i = df_dev["date"].searchsorted(val_month_end)

    train_indices = np.arange(train_start_i,train_end_i)
    val_indices = np.arange(train_end_i,val_end_i)

    custom_folds.append((train_indices,val_indices))

  def objective(trial):
    hidden_layers = trial.suggest_int("hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_categorical("neurons_per_layer", [16, 32, 64])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [1024, 2048, 4096])
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)

    fold_epochs = []
    fold_errors = []

    for fold_id, (train_i, val_i) in enumerate(custom_folds):
      tf.keras.backend.clear_session()

      X_train, X_val = tf.gather(X_dev, train_i), tf.gather(X_dev, val_i)
      y_train, y_val = tf.gather(y_dev, train_i), tf.gather(y_dev, val_i)

      model = tf.keras.Sequential()
      model.add(tf.keras.Input(shape=(X_train.shape[1],)))

      for i in range(hidden_layers):
        model.add(tf.keras.layers.Dense(neurons_per_layer))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation("relu"))
        model.add(tf.keras.layers.Dropout(dropout_rate))

      model.add(tf.keras.layers.Dense(1, activation="linear"))

      model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="mae")

      early_stopper = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
      callbacks = [early_stopper]
      if fold_id == 0:
        callbacks.append(TFKerasPruningCallback(trial, "val_loss"))

      history = model.fit(
          X_train, y_train,
          validation_data=(X_val, y_val),
          batch_size=batch_size,
          epochs=100,
          callbacks=callbacks,
          verbose=0
      )

      best_epoch = np.argmin(history.history["val_loss"]) + 1
      fold_epochs.append(best_epoch)

      mae = model.evaluate(X_val, y_val, verbose=0)
      fold_errors.append(mae)

    trial.set_user_attr("optimal_epochs", round(np.mean(fold_epochs)))

    return np.mean(fold_errors)

  ram_storage = optuna.storages.InMemoryStorage()

  study = optuna.create_study(
      study_name=f"NN_{dataset_name}",
      storage=ram_storage,
      load_if_exists=True,
      direction="minimize",
      pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=10)
  )

  tuning_start = time.perf_counter()
  study.optimize(objective, n_trials=50)
  tuning_time = time.perf_counter() - tuning_start

  optuna.copy_study(
      from_study_name=f"NN_{dataset_name}",
      from_storage=ram_storage,
      to_storage=DataBase_URL,
      to_study_name=f"NN_{dataset_name}"
  )

  best_params = study.best_params

  avg_best_epochs = study.best_trial.user_attrs["optimal_epochs"]
  best_params["epochs"] = int(np.round(avg_best_epochs))

  tf.keras.backend.clear_session()

  final_model = tf.keras.Sequential()
  final_model.add(tf.keras.Input(shape=(X_dev.shape[1],)))

  for i in range(best_params["hidden_layers"]):
    final_model.add(tf.keras.layers.Dense(best_params["neurons_per_layer"]))
    final_model.add(tf.keras.layers.BatchNormalization())
    final_model.add(tf.keras.layers.Activation("relu"))
    final_model.add(tf.keras.layers.Dropout(best_params["dropout_rate"]))

  final_model.add(tf.keras.layers.Dense(1, activation="linear"))

  final_model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["learning_rate"]),
      loss="mae"
  )

  training_start = time.perf_counter()

  history = final_model.fit(
      X_dev, y_dev,
      batch_size=best_params["batch_size"],
      epochs=best_params["epochs"],
      verbose=0
  )

  training_time = time.perf_counter() - training_start

  final_model.save(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/{dataset_name}_model.keras")

  testing_start = time.perf_counter()

  test_preds = final_model.predict(X_test)

  test_preds = test_preds.flatten()

  final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
  final_test_mae = mean_absolute_error(y_test, test_preds)

  testing_time = time.perf_counter() - testing_start

  history_df = pd.DataFrame(history.history)
  history_df.to_csv(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/{dataset_name}_history.csv", index=False)

  result_dict = {
      "Dataset": [dataset_name],
      "RMSE": [final_test_rmse],
      "MAE": [final_test_mae],
      "Tuning_Time_sec": [tuning_time],
      "Training_Time_sec": [training_time],
      "Testing_Time_sec": [testing_time]
  }

  df_iteration = pd.DataFrame(result_dict)
  csv_path = "drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/NeuralNetworkResults.csv"

  if not os.path.isfile(csv_path):
    df_iteration.to_csv(csv_path, index=False)
  else:
    df_iteration.to_csv(csv_path, mode="a", header=False, index=False)

  #all_datasets_results.append(result_dict)

#df_final_results = pd.DataFrame(all_datasets_results)
#df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/NeuralNetworkResults.csv", index=False)

Using Colab cache for faster access to the 'nyc-taxi-data-2023-24-normalized' dataset.


[I 2026-09-05 09:51:34,801] A new study created in memory with name: NN_df_PUs
[I 2026-09-05 09:55:04,707] Trial 0 finished with value: 0.412826473514239 and parameters: {'hidden_layers': 2, 'neurons_per_layer': 64, 'learning_rate': 0.0078094475937122605, 'batch_size': 4096, 'dropout_rate': 0.4799201276962272}. Best is trial 0 with value: 0.412826473514239.
[I 2026-09-05 10:07:41,404] Trial 1 finished with value: 0.4881654779116313 and parameters: {'hidden_layers': 3, 'neurons_per_layer': 16, 'learning_rate': 6.37760470463931e-05, 'batch_size': 1024, 'dropout_rate': 0.44588602228761004}. Best is trial 0 with value: 0.412826473514239.
[I 2026-09-05 10:12:05,448] Trial 2 finished with value: 0.48760352035363513 and parameters: {'hidden_layers': 4, 'neurons_per_layer': 16, 'learning_rate': 0.002663764716399982, 'batch_size': 2048, 'dropout_rate': 0.44747556505361047}. Best is trial 0 with value: 0.412826473514239.
[I 2026-09-05 10:26:04,552] Trial 3 finished with value: 0.3981412500143051

69061/69061 ━━━━━━━━━━━━━━━━━━━━ 83s 1ms/step


[I 2026-09-05 14:21:36,007] A new study created in memory with name: NN_df_PUsWEATHER
[I 2026-09-05 14:24:58,954] Trial 0 finished with value: 0.45052915811538696 and parameters: {'hidden_layers': 1, 'neurons_per_layer': 16, 'learning_rate': 0.0037531313163557093, 'batch_size': 2048, 'dropout_rate': 0.22993093409440502}. Best is trial 0 with value: 0.45052915811538696.
[I 2026-09-05 14:39:37,390] Trial 1 finished with value: 0.4466167688369751 and parameters: {'hidden_layers': 1, 'neurons_per_layer': 32, 'learning_rate': 3.2522007251984825e-05, 'batch_size': 2048, 'dropout_rate': 0.14114167615528916}. Best is trial 1 with value: 0.4466167688369751.
[I 2026-09-05 15:07:13,496] Trial 2 finished with value: 0.3887782692909241 and parameters: {'hidden_layers': 4, 'neurons_per_layer': 64, 'learning_rate': 2.6779729562139006e-05, 'batch_size': 1024, 'dropout_rate': 0.32175268591323275}. Best is trial 2 with value: 0.3887782692909241.
[I 2026-09-05 15:16:31,448] Trial 3 finished with value: 0

69061/69061 ━━━━━━━━━━━━━━━━━━━━ 88s 1ms/step


[I 2026-09-05 20:00:38,978] A new study created in memory with name: NN_df_PUsCYCLt
[I 2026-09-05 20:11:13,977] Trial 0 finished with value: 0.3875483175118764 and parameters: {'hidden_layers': 4, 'neurons_per_layer': 64, 'learning_rate': 0.0001294796669106637, 'batch_size': 4096, 'dropout_rate': 0.4839503464548934}. Best is trial 0 with value: 0.3875483175118764.
[I 2026-09-05 20:16:36,474] Trial 1 finished with value: 0.3100408564011256 and parameters: {'hidden_layers': 4, 'neurons_per_layer': 64, 'learning_rate': 0.003931975581008922, 'batch_size': 4096, 'dropout_rate': 0.17324519703274505}. Best is trial 1 with value: 0.3100408564011256.
[I 2026-09-05 20:23:30,910] Trial 2 finished with value: 0.38108129302660626 and parameters: {'hidden_layers': 2, 'neurons_per_layer': 16, 'learning_rate': 0.006587080295406381, 'batch_size': 1024, 'dropout_rate': 0.14733582493059555}. Best is trial 1 with value: 0.3100408564011256.
[I 2026-09-05 20:34:00,999] Trial 3 finished with value: 0.5214999

69061/69061 ━━━━━━━━━━━━━━━━━━━━ 88s 1ms/step


DOsCYCLtWEATHER + PUsCYCLtWEATHER

In [ ]:
import pandas as pd
import numpy as np
import time
import optuna
import os
from optuna_integration import TFKerasPruningCallback
import kagglehub
import tensorflow as tf
from tensorflow.keras import mixed_precision
from sklearn.metrics import mean_absolute_error, mean_squared_error

mixed_precision.set_global_policy("mixed_float16")

path = kagglehub.dataset_download("florianhinrichsen/nyc-taxi-data-2023-24-normalized/version/3", force_download=True)

datasets = {f"{path}/hourlyDOs202324WEATHER_CYCLt_Normalized.parquet":"df_DOsWEATHER_CYCLt",
            f"{path}/hourlyPUs202324WEATHER_CYCLt_Normalized.parquet":"df_PUsWEATHER_CYCLt"}


first_block_end = pd.to_datetime("2023-07-01")
test_set_start = pd.to_datetime("2024-01-01")

DataBase_URL = "sqlite:///drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/FFNN_tuning.db"

all_datasets_results = []

for file in datasets:
  dataset_name = datasets[file]
  df = pd.read_parquet(file)
  df = df.rename(columns={df.columns[1]:"date"})
  df = df[df["year"].isin([2023, 2024])]
  df["year"] = df["year"].map({2023: 0, 2024: 1})
  df = df.drop(columns=["DOLocationID", "PULocationID"], errors="ignore")
  df = df.drop(columns=["precipitation"], errors='ignore')
  df = df.drop(columns=["total_amount"], errors="ignore")
  df = df.dropna()

  df_test = df[df["date"]>=test_set_start].copy()
  df_dev = df[df["date"]<test_set_start].copy()

  X_test = df_test.drop(columns=["trip_count","date"])
  y_test = df_test["trip_count"]

  X_dev = df_dev.drop(columns=["trip_count","date"])
  y_dev = df_dev["trip_count"]

  X_dev = tf.constant(X_dev.values, dtype=tf.float32)
  y_dev = tf.constant(y_dev.values, dtype=tf.float32)

  custom_folds = []
  for month in range(6):
    val_month_start = first_block_end + pd.DateOffset(months=month)
    val_month_end = val_month_start + pd.DateOffset(months=1)

    train_month_start = val_month_start - pd.DateOffset(months=6)

    train_start_i = df_dev["date"].searchsorted(train_month_start)
    train_end_i = df_dev["date"].searchsorted(val_month_start)
    val_end_i = df_dev["date"].searchsorted(val_month_end)

    train_indices = np.arange(train_start_i,train_end_i)
    val_indices = np.arange(train_end_i,val_end_i)

    custom_folds.append((train_indices,val_indices))

  def objective(trial):
    hidden_layers = trial.suggest_int("hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_categorical("neurons_per_layer", [16, 32, 64])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [1024, 2048, 4096])
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)

    fold_epochs = []
    fold_errors = []

    for fold_id, (train_i, val_i) in enumerate(custom_folds):
      tf.keras.backend.clear_session()

      X_train, X_val = tf.gather(X_dev, train_i), tf.gather(X_dev, val_i)
      y_train, y_val = tf.gather(y_dev, train_i), tf.gather(y_dev, val_i)

      model = tf.keras.Sequential()
      model.add(tf.keras.Input(shape=(X_train.shape[1],)))

      for i in range(hidden_layers):
        model.add(tf.keras.layers.Dense(neurons_per_layer))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation("relu"))
        model.add(tf.keras.layers.Dropout(dropout_rate))

      model.add(tf.keras.layers.Dense(1, activation="linear"))

      model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="mae")

      early_stopper = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
      callbacks = [early_stopper]
      if fold_id == 0:
        callbacks.append(TFKerasPruningCallback(trial, "val_loss"))

      history = model.fit(
          X_train, y_train,
          validation_data=(X_val, y_val),
          batch_size=batch_size,
          epochs=100,
          callbacks=callbacks,
          verbose=0
      )

      best_epoch = np.argmin(history.history["val_loss"]) + 1
      fold_epochs.append(best_epoch)

      mae = model.evaluate(X_val, y_val, verbose=0)
      fold_errors.append(mae)

    trial.set_user_attr("optimal_epochs", round(np.mean(fold_epochs)))

    return np.mean(fold_errors)

  ram_storage = optuna.storages.InMemoryStorage()

  study = optuna.create_study(
      study_name=f"NN_{dataset_name}",
      storage=ram_storage,
      load_if_exists=True,
      direction="minimize",
      pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=10)
  )

  tuning_start = time.perf_counter()
  study.optimize(objective, n_trials=50)
  tuning_time = time.perf_counter() - tuning_start

  optuna.copy_study(
      from_study_name=f"NN_{dataset_name}",
      from_storage=ram_storage,
      to_storage=DataBase_URL,
      to_study_name=f"NN_{dataset_name}"
  )

  best_params = study.best_params

  avg_best_epochs = study.best_trial.user_attrs["optimal_epochs"]
  best_params["epochs"] = int(np.round(avg_best_epochs))

  tf.keras.backend.clear_session()

  final_model = tf.keras.Sequential()
  final_model.add(tf.keras.Input(shape=(X_dev.shape[1],)))

  for i in range(best_params["hidden_layers"]):
    final_model.add(tf.keras.layers.Dense(best_params["neurons_per_layer"]))
    final_model.add(tf.keras.layers.BatchNormalization())
    final_model.add(tf.keras.layers.Activation("relu"))
    final_model.add(tf.keras.layers.Dropout(best_params["dropout_rate"]))

  final_model.add(tf.keras.layers.Dense(1, activation="linear"))

  final_model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["learning_rate"]),
      loss="mae"
  )

  training_start = time.perf_counter()

  history = final_model.fit(
      X_dev, y_dev,
      batch_size=best_params["batch_size"],
      epochs=best_params["epochs"],
      verbose=0
  )

  training_time = time.perf_counter() - training_start

  final_model.save(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/{dataset_name}_model.keras")

  testing_start = time.perf_counter()

  test_preds = final_model.predict(X_test)

  test_preds = test_preds.flatten()

  final_test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
  final_test_mae = mean_absolute_error(y_test, test_preds)

  testing_time = time.perf_counter() - testing_start

  history_df = pd.DataFrame(history.history)
  history_df.to_csv(f"drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/{dataset_name}_history.csv", index=False)

  result_dict = {
      "Dataset": [dataset_name],
      "RMSE": [final_test_rmse],
      "MAE": [final_test_mae],
      "Tuning_Time_sec": [tuning_time],
      "Training_Time_sec": [training_time],
      "Testing_Time_sec": [testing_time]
  }

  df_iteration = pd.DataFrame(result_dict)
  csv_path = "drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network_new/NeuralNetworkResults.csv"

  if not os.path.isfile(csv_path):
    df_iteration.to_csv(csv_path, index=False)
  else:
    df_iteration.to_csv(csv_path, mode="a", header=False, index=False)

  #all_datasets_results.append(result_dict)

#df_final_results = pd.DataFrame(all_datasets_results)
#df_final_results.to_csv("drive/MyDrive/BT_Florian_2026/Trained Models/Neural Network/NeuralNetworkResults.csv", index=False)

Using Colab cache for faster access to the 'nyc-taxi-data-2023-24-normalized' dataset.


[I 2026-09-06 07:35:35,542] A new study created in memory with name: NN_df_DOsWEATHER_CYCLt
[I 2026-09-06 07:42:10,468] Trial 0 finished with value: 0.3950924724340439 and parameters: {'hidden_layers': 3, 'neurons_per_layer': 32, 'learning_rate': 0.0042010326526546685, 'batch_size': 1024, 'dropout_rate': 0.3156414266759249}. Best is trial 0 with value: 0.3950924724340439.
[I 2026-09-06 07:57:28,359] Trial 1 finished with value: 0.4293707460165024 and parameters: {'hidden_layers': 2, 'neurons_per_layer': 64, 'learning_rate': 3.70293205353574e-05, 'batch_size': 2048, 'dropout_rate': 0.4927285229638767}. Best is trial 0 with value: 0.3950924724340439.
[I 2026-09-06 08:04:11,126] Trial 2 finished with value: 0.36452074348926544 and parameters: {'hidden_layers': 4, 'neurons_per_layer': 64, 'learning_rate': 0.0007566016342846217, 'batch_size': 2048, 'dropout_rate': 0.33156068126736543}. Best is trial 2 with value: 0.36452074348926544.
[I 2026-09-06 08:06:45,183] Trial 3 finished with value: 

69629/69629 ━━━━━━━━━━━━━━━━━━━━ 87s 1ms/step


[I 2026-09-06 14:05:59,994] A new study created in memory with name: NN_df_PUsWEATHER_CYCLt
[I 2026-09-06 14:12:00,676] Trial 0 finished with value: 0.37695539991060895 and parameters: {'hidden_layers': 5, 'neurons_per_layer': 32, 'learning_rate': 0.0029288076949064096, 'batch_size': 2048, 'dropout_rate': 0.1453854672219827}. Best is trial 0 with value: 0.37695539991060895.
[I 2026-09-06 14:15:13,527] Trial 1 finished with value: 0.42011281351248425 and parameters: {'hidden_layers': 1, 'neurons_per_layer': 64, 'learning_rate': 0.00589221278561833, 'batch_size': 2048, 'dropout_rate': 0.18862767259063423}. Best is trial 0 with value: 0.37695539991060895.
[I 2026-09-06 14:33:57,036] Trial 2 finished with value: 0.37669505675633747 and parameters: {'hidden_layers': 4, 'neurons_per_layer': 64, 'learning_rate': 9.496662603064735e-05, 'batch_size': 1024, 'dropout_rate': 0.3654309954450927}. Best is trial 2 with value: 0.37669505675633747.
[I 2026-09-06 14:44:47,769] Trial 3 finished with valu

69061/69061 ━━━━━━━━━━━━━━━━━━━━ 90s 1ms/step
